In [0]:
dbutils.library.restartPython()

In [0]:
MODELS = [
    "gpt-4.1",
    "gpt-4o",
    "gpt-4.1-mini"
]

# Clean up existing widgets if re-running the notebook
dbutils.widgets.removeAll()

# Evaluator model (single selection)
dbutils.widgets.dropdown(
    name="Weather_Model",
    defaultValue="gpt-4.1-mini",
    choices=MODELS,
    label="Weather_Model"
)

In [0]:
model = dbutils.widgets.get("Weather_Model")

In [0]:
import sys
import asyncio
import os
import gradio as gr
from IPython.display import display, HTML
from agents import gen_trace_id, trace, Runner
from dotenv import load_dotenv
load_dotenv()

sys.path.append("/Workspace/Users/youruser@gmail.com/Weather_Assistant_AI_Agent/weather_alert_flow/src")

from Agents import weather_agent
from Weather_Tools import (
    resolve_location, get_current_weather, get_weather_forecast,
    get_weather_report, get_regional_weather, get_representative_locations
)
from Email_Tool import send_html_email

In [0]:
tools = [
        resolve_location,
        get_current_weather,
        get_weather_forecast,
        get_weather_report,
        get_regional_weather,
        get_representative_locations]
weather_agent = weather_agent(tools, model)

In [0]:
async def weather_workflow(user_input):

    yield "🌦️ Calling weather agent..."

    yield "🔍 Running guardrail validation..."

    try:
        trace_id = gen_trace_id()
        with trace("weather_report_agent", trace_id=trace_id):
            weather_result = await Runner.run(weather_agent,
                        f"""{user_input}""")
            yield f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}"
    except Exception as e:
        yield f"Hello, Im Weather assistant. I will help you with weather information. But I need to know your location. Please provide your location."
        # yield f"{e}"
        return

    yield "✅ Weather data received."

    yield "✉️ Creating email content..."


    final_output = f"""
        ## Subject
        {weather_result.final_output.subject}

        ## Email Body

        {weather_result.final_output.html_body}
        """

    yield final_output

In [0]:
async def run_weather_workflow(user_input):

    async for update in weather_workflow(
        f"""{user_input}"""
    ):
        # print(update)
        yield update

In [0]:
# user_input = "will tomorrow rain in hyderabad"
# await run_weather_workflow(user_input)

In [0]:
with gr.Blocks() as demo:
    gr.Markdown("# 🌦️ Weather Alert Assistant")

    query = gr.Textbox(label="Weather Query")

    result = gr.HTML(label="Result")

    submit = gr.Button("Run")

    submit.click(
        fn=run_weather_workflow,
        inputs=query,
        outputs=result
    )

demo.queue().launch(share=True, debug=True)